In [41]:
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from pathlib import Path
from PIL import Image

In [42]:
DATASET_PATH = Path(r"C:\Users\jessi\OneDrive\Desktop\aiims2")
manifest     = pd.read_csv(DATASET_PATH / 'processed_manifest.csv')

# Use GPU if available, otherwise CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print(f"Total cases: {len(manifest)}")

Using device: cpu
Total cases: 780


In [43]:
# Load pretrained ConvNeXt-Tiny
# Remove the final classification head — we want embeddings
convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
convnext.classifier = torch.nn.Identity()  # replace classifier with identity (no-op)
convnext = convnext.to(DEVICE)
convnext.eval()

print("ConvNeXt-Tiny loaded — output embedding size: 768")

ConvNeXt-Tiny loaded — output embedding size: 768


In [44]:
# ResNet expects 224x224 RGB images normalized with ImageNet stats
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

def to_rgb_tensor(crop_array):
    """
    Takes a 2D numpy array (grayscale, normalized float),
    converts to uint8, then to 3-channel RGB PIL image,
    then applies transform for ResNet input.
    """
    # Rescale to 0-255
    arr = crop_array.copy()
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255
    arr = arr.astype(np.uint8)

    # Convert grayscale to RGB by stacking 3 channels
    pil_img = Image.fromarray(arr).convert('RGB')

    return transform(pil_img).unsqueeze(0).to(DEVICE)  # add batch dim

In [45]:
def get_bounding_box(mask, padding=10):
    """
    Returns (row_min, row_max, col_min, col_max)
    of the bounding box around a binary mask, with optional padding.
    """
    rows = np.where(mask.sum(axis=1) > 0)[0]
    cols = np.where(mask.sum(axis=0) > 0)[0]

    if len(rows) == 0 or len(cols) == 0:
        return None  # empty mask

    h, w = mask.shape
    r_min = max(0, rows.min() - padding)
    r_max = min(h, rows.max() + padding)
    c_min = max(0, cols.min() - padding)
    c_max = min(w, cols.max() + padding)

    return r_min, r_max, c_min, c_max


def extract_crops(image, masks):
    """
    Returns 4 crops as numpy arrays:
    - tumor crop (tight bounding box around tumor)
    - perilesion crop (tumor + 15px ring bounding box)
    - posterior crop (region below tumor)
    - whole image
    """
    crops = {}

    # 1. Tumor crop
    bbox = get_bounding_box(masks['tumor_mask'], padding=5)
    if bbox:
        r0, r1, c0, c1 = bbox
        crops['tumor'] = image[r0:r1, c0:c1]
    else:
        crops['tumor'] = image  # fallback to whole image

    # 2. Perilesion crop (tumor + ring_15px combined)
    combined_mask = np.clip(masks['tumor_mask'] + masks['ring_15px'], 0, 1)
    bbox = get_bounding_box(combined_mask, padding=10)
    if bbox:
        r0, r1, c0, c1 = bbox
        crops['perilesion'] = image[r0:r1, c0:c1]
    else:
        crops['perilesion'] = image

    # 3. Posterior crop
    bbox = get_bounding_box(masks['posterior_mask'], padding=0)
    if bbox:
        r0, r1, c0, c1 = bbox
        crops['posterior'] = image[r0:r1, c0:c1]
    else:
        crops['posterior'] = image

    # 4. Whole image
    crops['whole'] = image

    return crops

In [46]:
def get_embedding(model, crop_array):
    tensor = to_rgb_tensor(crop_array)
    with torch.no_grad():
        embedding = model(tensor)
    return embedding.squeeze().cpu().numpy()  # shape: (768,)

In [50]:
sample_row = manifest.iloc[0]
sample     = np.load(sample_row['npz_path'])

image = sample['image']
masks = {
    'tumor_mask'     : sample['tumor_mask'],
    'ring_5px'       : sample['ring_5px'],
    'ring_15px'      : sample['ring_15px'],
    'posterior_mask' : sample['posterior_mask'],
    'skin_mask'      : sample['skin_mask'],
    'background_mask': sample['background_mask'],
}

crops = extract_crops(image, masks)

print("Crop shapes:")
for name, crop in crops.items():
    print(f"  {name}: {crop.shape}")

# Test embedding on tumor crop
emb = get_embedding(convnext, crops[crop_name])
print(f"\nTumor embedding shape: {emb.shape}")
print(f"First 5 values: {emb[:5]}")

Crop shapes:
  tumor: (23, 32)
  perilesion: (63, 72)
  posterior: (168, 255)
  whole: (256, 256)

Tumor embedding shape: (768,)
First 5 values: [ 0.06517565 -0.0800712  -0.43099135  0.11662378 -0.0655417 ]


In [51]:
CROP_NAMES = ['tumor', 'perilesion', 'posterior', 'whole']

all_records = []

for idx, row in manifest.iterrows():
    if idx % 50 == 0:
        print(f"Processing {idx}/{len(manifest)}...")

    try:
        sample = np.load(row['npz_path'])
        image  = sample['image']
        masks  = {
            'tumor_mask'     : sample['tumor_mask'],
            'ring_5px'       : sample['ring_5px'],
            'ring_15px'      : sample['ring_15px'],
            'posterior_mask' : sample['posterior_mask'],
            'skin_mask'      : sample['skin_mask'],
            'background_mask': sample['background_mask'],
        }

        crops = extract_crops(image, masks)

        record = {
            'case_id': row['case_id'],
            'label'  : row['label'],
        }

        for crop_name in CROP_NAMES:
            emb = get_embedding(convnext, crops[crop_name])
            for i, val in enumerate(emb):
                record[f'{crop_name}_{i}'] = float(val)

        all_records.append(record)

    except Exception as e:
        print(f"ERROR on {row['case_id']}: {e}")

print(f"\nDone! Total records: {len(all_records)}")

Processing 0/780...
Processing 50/780...
Processing 100/780...
Processing 150/780...
Processing 200/780...
Processing 250/780...
Processing 300/780...
Processing 350/780...
Processing 400/780...
Processing 450/780...
Processing 500/780...
Processing 550/780...
Processing 600/780...
Processing 650/780...
Processing 700/780...
Processing 750/780...

Done! Total records: 780


In [52]:
df_deep = pd.DataFrame(all_records)
df_deep.to_csv(DATASET_PATH / 'features_deep.csv', index=False)

print(f"Saved features_deep.csv")
print(f"Shape: {df_deep.shape}")
print(f"Columns: case_id, label + {len(df_deep.columns)-2} embedding dims")
df_deep.iloc[:3, :10]  # preview first 10 columns

Saved features_deep.csv
Shape: (780, 3074)
Columns: case_id, label + 3072 embedding dims


,case_id,label,tumor_0,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7
0,benign (1),benign,-0.140227,0.199341,0.014288,-0.093048,0.300193,-0.061440,-0.245403,0.049486
1,benign (10),benign,-0.004000,0.183299,-0.220773,0.085275,0.018301,-0.010775,-0.196513,-0.021403
2,benign (100),benign,0.047933,0.075989,-0.036841,0.121892,-0.058393,-0.014752,0.028765,0.063016


In [53]:
df_texture = pd.read_csv(DATASET_PATH / 'features_texture.csv')

# Pivot texture features to wide format (one row per case)
df_texture_wide = df_texture.pivot_table(
    index=['case_id', 'label'],
    columns='region',
    values=[c for c in df_texture.columns
            if c not in ['case_id', 'label', 'region']],
    aggfunc='first'
).reset_index()

# Flatten column names: feature_region
df_texture_wide.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in df_texture_wide.columns
]

# Merge with deep features
df_combined = df_texture_wide.merge(df_deep, on=['case_id', 'label'], how='inner')

df_combined.to_csv(DATASET_PATH / 'features_combined.csv', index=False)

print(f"features_texture wide shape : {df_texture_wide.shape}")
print(f"features_deep shape         : {df_deep.shape}")
print(f"features_combined shape     : {df_combined.shape}")

features_texture wide shape : (780, 134)
features_deep shape         : (780, 3074)
features_combined shape     : (780, 3206)
